# Transformers and the modern architectures

Transformers (and attention) are the foundational advance that enabled Large Language Models (LLMs). Unlike a Deep Neural Network, there is a specific architecture that they adhere to that consists of an encoding and decoding component. 

![](https://miro.medium.com/v2/resize:fit:1400/format:webp/0*376uJu_fc_uR8H3X.png)

Some of the best accessible explanations on these different architectures can be found in the [HuggingFace tutorials](https://huggingface.co/learn/llm-course/en/chapter1/6). We will actually want to watch the encoder architecture video to get some greater context.

[Encoder](https://youtu.be/MUqNwgPjJvQ)

# And now the cheat sheet for our architecture choice

Based on what the text task is that we want to do, it's relatively straightforward what the optimal architecture choice would be. 

| Task	| Suggested Architecture	| Examples |
| :------- | :------: | -------: |
| Text classification (sentiment, topic)	| Encoder	| BERT, RoBERTa|
| Text generation (creative writing)	| Decoder	| GPT, LLaMA |
| Translation	| Encoder-Decoder	| T5, BART |
| Summarization	| Encoder-Decoder	| BART, T5 |
| Named entity recognition	| Encoder	| BERT, RoBERTa |
| Question answering (extractive)	| Encoder	| BERT, RoBERTa |
| Question answering (generative)	| Encoder-Decoder or Decoder	| T5, GPT |
| Conversational AI	| Decoder	| GPT, LLaMA |

What's the pattern here? 

# In Practice

HuggingFace is extremely easy to use in practice and you can start working in it immediately

In [1]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu
!pip install transformers

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download.pytorch.org/whl/cpu/torch-2.9.1%2Bcpu-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached https://download.pytorch.org/whl/sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
Using cached https://download.pytorch.org/whl/cpu/torch-2.9.1%2Bcpu-cp311-cp311-manylinux_2_28_x86_64.whl (184.5 MB)
Using cached https://download.pytorch.org/whl/sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached https://download.pytorch.org/whl/filelock-3.19.1-py3-none-any.whl (15 kB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.12.1
    Uninstalling sympy-1.12.1:
      Successfully uninstalled sympy-1.12.1
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2025.11.3-cp311-cp311-

In [1]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

classifier("I've been waiting for a HuggingFace course my whole life.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9598048329353333}]

Huggingface is built around the idea of **pipelines** that stitch together both *preprocessing* and *postprocessing*. You are **not** restricted to using the existing pipelines, you can make your own out of whatever models you choose (including those that you finetune!). But there are a large number of pre-built pipelines usiong pre-trained models that you can leverage out of the box like so:

**Text pipelines** 

* text-generation: Generate text from a prompt
* text-classification: Classify text into predefined categories
* summarization: Create a shorter version of a text while preserving key information
* translation: Translate text from one language to another
* zero-shot-classification: Classify text without prior training on specific labels
* feature-extraction: Extract vector representations of text


**Image pipelines**

* image-to-text: Generate text descriptions of images
* image-classification: Identify objects in an image
* object-detection: Locate and identify objects in images

**Audio pipelines**

* automatic-speech-recognition: Convert speech to text
* audio-classification: Classify audio into categories
* text-to-speech: Convert text to spoken audio

**Multimodal pipelines**

* image-text-to-text: Respond to an image based on a text prompt


Let's quickly try some more to explore.

In [2]:
summarizer = pipeline('summarization')

#We have to prep the data because it's limited to 1024 tokens
short_data = open('../../data/text_data/wiki_plaintext_articles/Adam_Smith.txt').read()[:1023]

summarizer( short_data )

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'summary_text': ' Adam Smith, FRSE, was a Scottish political economist and moral philosopher . He was born in Kirkcaldy, Fife, Scotland, in 1723 and died July 17, 1790 in Edinburgh, Edinburgh, Scotland . Adam Smith was an 18th-century philosopher and political economist .'}]

And probably more useful for us, we can demonstrate the zero-shot classification pipeline


In [9]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="NbAiLab/nb-bert-base-mnli")
sequence_to_classify = "Angela Merkel is a politician in Germany and leader of the CDU"
candidate_labels = ["politics", "economy", "entertainment", "environment"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)

print(output)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'sequence': 'Angela Merkel is a politician in Germany and leader of the CDU', 'labels': ['politics', 'economy', 'entertainment', 'environment'], 'scores': [0.6867387294769287, 0.12389851361513138, 0.10405152291059494, 0.08531123399734497]}


In [10]:
classifier( short_data, candidate_labels=["education", "politics", "business"], multi_label=False)

{'sequence': '   #copyright\n\nAdam Smith\n\n2007 Schools Wikipedia Selection. Related subjects: Economics; Historical\nfigures\n\n                            Western Philosophers\n   18th-century philosophy\n   (Modern Philosophy)\n   Adam Smith\n         Name:       Adam Smith\n        Birth:       June 5, 1723 (baptised) ( Kirkcaldy, Fife, Scotland)\n        Death:       July 17, 1790 (Edinburgh, Scotland)\n   School/tradition: Classical economics\n    Main interests:  Political philosophy, ethics, economics\n    Notable ideas:   Classical economics, modern free market, division of\n                     labour\n      Influences:    Aristotle, Hobbes, Locke, Mandeville, Hutcheson, Hume,\n                     Montesquieu\n      Influenced:    Malthus, Ricardo, Mill, Keynes, Marx, Engels, American\n                     Founding Fathers\n\n   Adam Smith, FRSE, (baptised and probably born June 5, 1723 O.S. ( June\n   16 N.S.) – July 17, 1790) was a Scottish political economist and moral\

At this point you've probably noticed that it can be **pretty slow**. That's because in Callisto (and honestly most people's laptops) we're stuck with using the CPU version of these models. LLMs truly need GPUs (graphics cards) to perform well -- **especially for training**, but also in inference when doing more intensive tasks. You can access GPU resources through the traditional ARCTIC environment (which we're not going to cover) or on external public clouds (Google Cloud/Colab, AWS, Azure, etc.). 

**tldr;** We can't fine tune a model on Callisto because there aren't GPUs. We can only, effectively, fine-tune a model on a compute resource that has GPU access. So towards that end we will switch to co-lab and I'll demonstrate how to set-up all of the interworking components/platforms to fine-tune a model with the huggingface supplied fine-tuning notebook.

[linking out to colab](https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/language_modeling.ipynb)